# Lab: deploy an application on Kubernetes

You deploy **DockerCoins**, a small application made of five containers, in three ways:
one command at a time, from a YAML file, and then scaled to several replicas.

| Part | What you do | Time |
|---|---|---|
| I | Deploy with `kubectl create` and `kubectl expose`, find why the worker fails | 20 min |
| II | Deploy the same application from `dockercoins.yaml` | 10 min |
| III | Scale the worker and watch the effect | 15 min |

Deliverable: this notebook with your answers to the 10 questions, and the output of the last `kubectl get all`.

![DockerCoins architecture](figures/dockercoins-architecture.svg)

How to read this lab: run the cells in order. A command that never returns (`port-forward`) goes in a **terminal**, not in a cell.

## Prerequisites

* Docker Desktop with Kubernetes enabled (Settings, Kubernetes, Enable), and `kubectl`
* Check that the cluster answers: the cell below must list one node with the status `Ready`

In [ ]:
!kubectl get nodes

## Setup

Resources live in a namespace of their own, so the cleanup at the end is one command.

In [ ]:
!kubectl create namespace dockercoins
!kubectl config set-context --current --namespace=dockercoins

In [ ]:
!kubectl config view --minify | grep namespace

## Part I: deploy with commands

### 1. Create the five Deployments

In [ ]:
!kubectl create deployment redis --image redis:7
!kubectl create deployment rng --image dockercoins/rng:v0.1
!kubectl create deployment hasher --image dockercoins/hasher:v0.1
!kubectl create deployment worker --image dockercoins/worker:v0.1
!kubectl create deployment webui --image dockercoins/webui:v0.1

The images download first. This command waits until every Deployment is available (up to 3 minutes), then look at what exists.

In [ ]:
!kubectl wait --for=condition=available deployment --all --timeout=180s

In [ ]:
!kubectl get pods,replicasets,deployments

**Question 1.** You created 5 Deployments. How many pods and how many ReplicaSets do you see, and who created the ReplicaSets? How is a pod name built?

**Your answer:**



### 2. The worker cannot find redis

The worker should be hashing. Read its logs.

In [ ]:
!kubectl logs deployment/worker --tail=5

**Question 2.** What error do you see, and why can the worker not reach redis although the redis pod is running?

**Your answer:**



### 3. Labels and selectors

Every pod got a label `app=<name>` from `kubectl create deployment`. A Service finds its pods through such a label.

In [ ]:
!kubectl get pods --show-labels

In [ ]:
!kubectl get pods --selector=app=worker

In [ ]:
!kubectl get pods --selector=app!=hasher

In [ ]:
!kubectl get pods -o jsonpath='{range .items[*]}{.metadata.name}{"  "}{.spec.containers[0].image}{"\n"}{end}'

**Question 3.** Write the command that lists all pods except `hasher` and `redis` (hint: a selector can combine two conditions with a comma).

**Your answer:**



### 4. Expose the services

`kubectl expose` creates a Service for a Deployment, with the right selector.

In [ ]:
!kubectl get services

In [ ]:
!kubectl expose deployment redis --port 6379
!kubectl expose deployment rng --port 80
!kubectl expose deployment hasher --port 80
!kubectl expose deployment webui --port 80

In [ ]:
!kubectl get services

In [ ]:
!kubectl describe service redis

**Question 4.** In the output of `describe service redis`, which line tells you which pods receive the traffic, and what do the `Endpoints` contain?

**Your answer:**



The worker retries on its own. It waits 10 seconds between attempts; check its logs again after 15 seconds. The error should be gone and the worker reports units of work done.

In [ ]:
!sleep 15 && kubectl logs deployment/worker --tail=5

### 5. Open the web interface

The `webui` Service is of type `ClusterIP`: it is only reachable **inside** the cluster. To reach it from your laptop, forward a local port.
Run this in a **terminal** and leave it running:

```shell
kubectl port-forward service/webui 8080:80
```

Then open http://localhost:8080/index.html. The graph shows the hashes per second. Stop the forward with Ctrl+C when you are done.

Other ways in: a Service of type `NodePort` opens a port on every node, and a type `LoadBalancer` asks the cloud for an external address.

**Question 5.** Before you used `port-forward`, http://localhost did not answer. Why? Give the type of the Service and what that type means.

**Your answer:**



### Clean up Part I

Delete the Services and Deployments (the ReplicaSets and pods go with them).

In [ ]:
!kubectl delete services,deployments --all

In [ ]:
!kubectl wait --for=delete pod --all --timeout=60s

In [ ]:
!kubectl get all

## Part II: deploy from a file

Instead of telling Kubernetes what to **do**, you describe what you **want** in YAML. `kubectl apply` makes the cluster match the file.
`kubectl` can write a first draft for you: `--dry-run=client -o yaml` prints the manifest and creates nothing.

In [ ]:
!kubectl create deployment hasher --image dockercoins/hasher:v0.1 --dry-run=client -o yaml

Open `dockercoins.yaml` in your editor. It holds the 5 Deployments and the 4 Services you created by hand. Find in it:

* the `selector` of a Service and the `labels` of the pod template it matches
* the `image` of the worker
* the `replicas` of the worker

In [ ]:
!kubectl apply -f dockercoins.yaml

In [ ]:
!kubectl wait --for=condition=available deployment --all --timeout=180s

In [ ]:
!kubectl get pods,services,deployments

**Question 6.** Run the `kubectl apply` cell a second time. What does each line say now, and why is that useful?

**Your answer:**



Now change the file: set `replicas: 2` for the `worker` Deployment, save, and apply again.

In [ ]:
!kubectl apply -f dockercoins.yaml

In [ ]:
!kubectl get pods --selector=app=worker

**Question 7.** Which line of the `apply` output changed, and how many worker pods do you have now?

**Your answer:**



**Question 8.** Part I and Part II give the same application. Give one advantage of each way of working.

**Your answer:**



Open the web interface again (`kubectl port-forward service/webui 8080:80` in a terminal, then http://localhost:8080/index.html) and note the speed with **one** worker.

## Part III: scale the worker

The worker is the part that does the work, so it is the one to scale.

In [ ]:
!kubectl get pods,replicasets,deployments

In [ ]:
!kubectl scale deployment worker --replicas 2

In [ ]:
!kubectl rollout status deployment/worker

In [ ]:
!kubectl get pods -o wide

Wait a minute and watch the speed in the web interface.

**Question 9.** What happens to the hashes per second with 2 workers, then with 3? What does that tell you about where the work is done?

**Your answer:**



In [ ]:
!kubectl scale deployment worker --replicas 3

In [ ]:
!kubectl rollout status deployment/worker

In [ ]:
!kubectl get pods -o wide

### Self-healing

Delete one worker pod by hand and watch what Kubernetes does.

In [ ]:
!kubectl delete $(kubectl get pods --selector=app=worker -o name | head -1)
!kubectl get pods --selector=app=worker

**Question 10.** You deleted one of the 3 worker pods. How many worker pods exist a few seconds later, and which object noticed the difference?

**Your answer:**



## Hand in and clean up

Run this cell **before** the cleanup and keep its output in the notebook: it is part of the deliverable.

In [ ]:
!kubectl get all

Delete the namespace (everything inside it goes with it) and put your context back on the default namespace.

In [ ]:
!kubectl delete namespace dockercoins
!kubectl config set-context --current --namespace=default

Hand in: this notebook with your answers and the output of `kubectl get all` above.